# 🌍 Landslide4Sense — EDA Completo + Pipeline de Entrenamiento
## Proyecto: Detección de Deslizamientos mediante Fine-Tuning de CNN Profundas

**Dataset:** Landslide4Sense (ISPRS Competition 2022)  
**Canales:** 14 (Sentinel-2, Sentinel-1 SAR, ALOS DEM, Pendiente)  
**Tarea:** Clasificación binaria de parches 128×128 (deslizamiento / no-deslizamiento)

---
### 📋 Estructura del Notebook
1. [Configuración del Entorno](#env)
2. [Carga y Validación del Dataset](#load)
3. [EDA — Estadísticas por Canal](#stats)
4. [EDA — Visualización de Muestras](#viz)
5. [EDA — Correlación entre Canales](#corr)
6. [EDA — Verificación de Leakage](#leak)
7. [Preprocesamiento y Augmentation](#preprocess)
8. [Modelos: ResNet-50, EfficientNet-B4, U-Net](#models)
9. [Entrenamiento con 5-Fold CV](#train)
10. [Evaluación y Métricas Finales](#eval)

## 1. Configuración del Entorno <a id='env'></a>

In [ ]:
# Instalar dependencias (ejecutar solo en Colab)
import subprocess, sys

packages = [
    'h5py', 'numpy', 'pandas', 'matplotlib', 'seaborn',
    'scipy', 'scikit-learn', 'albumentations',
    'torch', 'torchvision', 'segmentation-models-pytorch',
    'tqdm', 'Pillow'
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print("✅ Dependencias instaladas")

In [ ]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import h5py
from scipy import stats as scipy_stats
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, roc_auc_score, precision_score,
                              recall_score, confusion_matrix, roc_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as tv_models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm
import random

warnings.filterwarnings('ignore')

# ─── REPRODUCIBILIDAD ────────────────────────────────────────────────
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Entorno configurado | Dispositivo: {DEVICE} | Seed: {SEED}")

## 2. Carga y Validación del Dataset <a id='load'></a>

### 📁 Estructura esperada del dataset
```
Landslide4Sense/
├── TrainData/
│   ├── img/        ← 3799 archivos .h5 (imagen 128×128×14)
│   └── mask/       ← 3799 archivos .h5 (máscara binaria 128×128)
├── ValData/
│   ├── img/        ← 245 archivos .h5
│   └── mask/       ← 245 archivos .h5
└── TestData/
    └── img/        ← 800 archivos .h5 (SIN etiquetas — competición)
```

> **Nota:** Si solo tienes TestData, el EDA de distribución de canales es completamente válido.
> El análisis por clase requiere TrainData/mask.


In [ ]:
# ─── CONFIGURAR RUTAS ─────────────────────────────────────────────────────
# En Colab: subir el dataset o montar Google Drive
# AJUSTA esta ruta a la ubicación de tu dataset:
DATA_ROOT = '/content/Landslide4Sense'   # ← MODIFICA según tu configuración

# Si usas Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = '/content/drive/MyDrive/Landslide_ML'

TRAIN_IMG_DIR  = os.path.join(DATA_ROOT, 'TrainData', 'img')
TRAIN_MASK_DIR = os.path.join(DATA_ROOT, 'TrainData', 'mask')
VAL_IMG_DIR    = os.path.join(DATA_ROOT, 'ValData',   'img')
VAL_MASK_DIR   = os.path.join(DATA_ROOT, 'ValData',   'mask')
TEST_IMG_DIR   = os.path.join(DATA_ROOT, 'TestData',  'img')

# Auto-detectar qué particiones están disponibles
available = {}
for name, path in [('train', TRAIN_IMG_DIR), ('val', VAL_IMG_DIR), ('test', TEST_IMG_DIR)]:
    files = sorted(glob.glob(os.path.join(path, '*.h5'))) if os.path.exists(path) else []
    available[name] = files
    print(f"  {'✅' if files else '❌'} {name:6s}: {len(files):4d} archivos en {path}")

print(f"\n{'✅' if available['test'] else '⚠️'} EDA de canales: disponible con TestData")
print(f"{'✅' if available['train'] else '⚠️'} EDA por clase: requiere TrainData + masks")

In [ ]:
# ─── INSPECCIONAR UN ARCHIVO H5 ───────────────────────────────────────────
sample_files = available.get('test') or available.get('train') or []
if not sample_files:
    raise FileNotFoundError("No se encontró ninguna partición del dataset. Verifica DATA_ROOT.")

with h5py.File(sample_files[0], 'r') as f:
    print("Claves en el archivo H5:", list(f.keys()))
    for key in f.keys():
        arr = f[key][:]
        print(f"  '{key}': shape={arr.shape}, dtype={arr.dtype}, "
              f"min={arr.min():.4f}, max={arr.max():.4f}")

# Verificar si hay máscaras disponibles
if available['train']:
    mask_files = sorted(glob.glob(os.path.join(TRAIN_MASK_DIR, '*.h5')))
    if mask_files:
        with h5py.File(mask_files[0], 'r') as f:
            for key in f.keys():
                arr = f[key][:]
                print(f"  Máscara '{key}': shape={arr.shape}, valores únicos={np.unique(arr)}")

## 3. EDA — Estadísticas por Canal <a id='stats'></a>

Los 14 canales provienen de 4 fuentes diferentes con escalas distintas:
| Canales | Fuente | Descripción |
|---------|--------|-------------|
| 0–6 | Sentinel-2 | Bandas ópticas (Azul, Verde, Rojo, NIR, NIR-A, SWIR1, SWIR2) |
| 7–8 | Sentinel-1 | SAR (VV, VH) |
| 9–10 | ALOS PALSAR | DEM Elevación, Pendiente (Slope) |
| 11–13 | Sentinel-2 | Red-Edge (B5, B6, B7) |


In [ ]:
# ─── NOMBRES ESTÁNDAR DE CANALES ──────────────────────────────────────────
CHANNEL_NAMES = [
    'S2-B2 Azul', 'S2-B3 Verde', 'S2-B4 Rojo', 'S2-B8 NIR',
    'S2-B8A NIR-A', 'S2-B11 SWIR1', 'S2-B12 SWIR2',
    'S1-VV SAR', 'S1-VH SAR',
    'ALOS DEM', 'DEM Slope',
    'S2-B5 RedEdge1', 'S2-B6 RedEdge2', 'S2-B7 RedEdge3'
]
CHANNEL_GROUPS = {
    'Sentinel-2 Óptico': list(range(7)),
    'Sentinel-1 SAR': [7, 8],
    'ALOS DEM': [9, 10],
    'S2 Red-Edge': [11, 12, 13]
}

# ─── CÓMPUTO INCREMENTAL DE ESTADÍSTICAS ──────────────────────────────────
# (evita cargar todo en memoria simultáneamente)
def compute_channel_stats(img_files, n_sample=100, subsample_step=8):
    """Calcula estadísticas por canal de forma incremental (Welford's algorithm)."""
    files_sample = img_files[::max(1, len(img_files)//n_sample)][:n_sample]
    
    sum1  = np.zeros(14)
    sum2  = np.zeros(14)
    n_pix = 0
    c_min = np.full(14,  np.inf)
    c_max = np.full(14, -np.inf)
    hist_samples = {c: [] for c in range(14)}
    
    for fp in tqdm(files_sample, desc='Calculando estadísticas', leave=False):
        with h5py.File(fp, 'r') as hf:
            arr = hf['img'][:].astype(np.float32)   # (128, 128, 14)
        flat = arr.reshape(-1, 14)                   # (16384, 14)
        n_pix += flat.shape[0]
        sum1  += flat.sum(0)
        sum2  += (flat ** 2).sum(0)
        c_min  = np.minimum(c_min, flat.min(0))
        c_max  = np.maximum(c_max, flat.max(0))
        for c in range(14):
            hist_samples[c].extend(flat[::subsample_step, c].tolist())
    
    mean = sum1 / n_pix
    std  = np.sqrt(np.maximum(sum2 / n_pix - mean ** 2, 0))
    
    stats_df = pd.DataFrame({
        'Canal': range(14),
        'Nombre': CHANNEL_NAMES,
        'Min':   c_min,
        'Max':   c_max,
        'Media': mean,
        'Std':   std,
    })
    return stats_df, hist_samples, len(files_sample)

# Usar TestData si TrainData no está disponible
eda_files = available['train'] or available['test']
print(f"Calculando estadísticas sobre {'TrainData' if available['train'] else 'TestData'}...")
stats_df, hist_samples, n_files = compute_channel_stats(eda_files)

print(f"\n✅ Estadísticas calculadas sobre {n_files} imágenes ({n_files*128*128:,} píxeles por canal)\n")
print(stats_df.to_string(index=False, float_format='%.4f'))

In [ ]:
# ─── TABLA RESUMEN ENRIQUECIDA ─────────────────────────────────────────────
# Añadir percentiles
for p_val, p_name in [(25,'P25'), (50,'P50'), (75,'P75')]:
    stats_df[p_name] = [float(np.percentile(hist_samples[c], p_val))
                        for c in range(14)]

# Coeficiente de variación (CV = std/mean * 100)
stats_df['CV%'] = (stats_df['Std'] / (stats_df['Media'] + 1e-8) * 100).round(1)

print("Tabla estadística completa por canal:")
print(stats_df[['Canal','Nombre','Min','Max','Media','Std','P25','P50','P75','CV%']]
      .to_string(index=False, float_format='%.4f'))

## 4. EDA — Visualización de Muestras <a id='viz'></a>

In [ ]:
def load_patch(filepath):
    with h5py.File(filepath, 'r') as hf:
        return hf['img'][:].astype(np.float32)

def normalize_for_display(arr_2d, percentile=99):
    """Normaliza un canal a [0,1] para visualización.""""
    a = arr_2d.copy()
    mask = a > 0.001
    if mask.any():
        p_high = np.percentile(a[mask], percentile)
        a = np.clip(a / max(p_high, 1e-6), 0, 1)
    return a

def plot_patch_all_views(filepath, title=''):
    """Muestra un parche en 6 vistas: RGB, Falso Color NIR, SWIR, SAR VV, DEM, Slope."""
    arr = load_patch(filepath)
    
    views = {
        'RGB (B4-B3-B2)': np.stack([normalize_for_display(arr[:,:,2]),
                                     normalize_for_display(arr[:,:,1]),
                                     normalize_for_display(arr[:,:,0])], axis=-1),
        'Falso Color NIR
(B8-B3-B2)': np.stack([normalize_for_display(arr[:,:,3]),
                                                   normalize_for_display(arr[:,:,1]),
                                                   normalize_for_display(arr[:,:,0])], axis=-1),
        'SWIR Compuesto
(B12-B8A-B4)': np.stack([normalize_for_display(arr[:,:,6]),
                                                    normalize_for_display(arr[:,:,4]),
                                                    normalize_for_display(arr[:,:,2])], axis=-1),
    }
    gray_views = {
        'SAR VV
(Sentinel-1)': (normalize_for_display(arr[:,:,7]), 'gray'),
        'DEM Elevación
(ALOS PALSAR)': (normalize_for_display(arr[:,:,9]), 'terrain'),
        'Pendiente (°)
(DEM Slope)': (normalize_for_display(arr[:,:,10]), 'hot'),
    }
    
    fig, axes = plt.subplots(2, 3, figsize=(13, 9))
    fig.suptitle(title, fontsize=12, fontweight='bold')
    
    for ax, (name, rgb) in zip(axes[0], views.items()):
        ax.imshow(rgb)
        ax.set_title(name, fontsize=9, fontweight='bold')
        ax.axis('off')
    
    for ax, (name, (img, cmap)) in zip(axes[1], gray_views.items()):
        im = ax.imshow(img, cmap=cmap)
        ax.set_title(name, fontsize=9, fontweight='bold')
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()

# Visualizar 3 parches representativos
sample_files_viz = eda_files[::len(eda_files)//3][:3]
for i, fp in enumerate(sample_files_viz):
    plot_patch_all_views(fp, f'Parche #{i+1}: {os.path.basename(fp)}')

In [ ]:
# ─── HISTOGRAMAS POR CANAL ─────────────────────────────────────────────────
fig, axes = plt.subplots(4, 4, figsize=(17, 13))
fig.suptitle('Distribución de Valores por Canal\n(densidad, excluyendo ceros)', 
             fontsize=13, fontweight='bold')

group_colors = {
    'Sentinel-2 Óptico': '#3498db',
    'Sentinel-1 SAR':    '#e74c3c',
    'ALOS DEM':          '#2ecc71',
    'S2 Red-Edge':       '#9b59b6'
}

def get_channel_group(ch_idx):
    for group, channels in CHANNEL_GROUPS.items():
        if ch_idx in channels:
            return group
    return 'Otro'

for c in range(14):
    ax = axes[c // 4, c % 4]
    vals = np.array(hist_samples[c])
    vals_nonzero = vals[vals > 0.001]
    
    group = get_channel_group(c)
    color = group_colors[group]
    
    ax.hist(vals_nonzero, bins=60, color=color, alpha=0.75, 
            density=True, edgecolor='none')
    
    mu = stats_df.loc[c, 'Media']
    me = stats_df.loc[c, 'P50']
    sd = stats_df.loc[c, 'Std']
    
    ax.axvline(mu, color='red', lw=1.8, ls='--', alpha=0.9, label='Media')
    ax.axvline(me, color='navy', lw=1.8, ls=':', alpha=0.9, label='Mediana')
    
    ax.set_title(f'Ch{c}: {CHANNEL_NAMES[c]}', fontsize=8.5, fontweight='bold', color=color)
    ax.set_xlabel('Valor', fontsize=7)
    ax.set_ylabel('Densidad', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6, framealpha=0.7)
    ax.text(0.97, 0.96, f'μ={mu:.3f}\nσ={sd:.3f}', transform=ax.transAxes,
            fontsize=7.5, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))
    ax.grid(axis='y', alpha=0.3, lw=0.5)

# Ocultar el último panel vacío (solo 14 canales, 4×4=16 paneles)
axes[3, 2].set_visible(False)
axes[3, 3].set_visible(False)

# Leyenda global de grupos
from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=c, label=g) for g, c in group_colors.items()]
fig.legend(handles=legend_patches, loc='lower right', ncol=2, fontsize=9, 
           bbox_to_anchor=(0.98, 0.01))

plt.tight_layout()
plt.savefig('fig_histogramas_canales.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Histogramas guardados")

## 5. EDA — Correlación entre Canales <a id='corr'></a>

In [ ]:
# ─── MATRIZ DE CORRELACIÓN DE PEARSON ─────────────────────────────────────
def compute_correlation_matrix(img_files, n_imgs=30):
    """Calcula correlación de Pearson entre los 14 canales.""""
    flat_list = []
    for fp in tqdm(img_files[:n_imgs], desc='Cargando para correlación', leave=False):
        with h5py.File(fp, 'r') as hf:
            arr = hf['img'][:].astype(np.float32)
        flat_list.append(arr.reshape(-1, 14))
    flat = np.vstack(flat_list)   # (n_imgs * 16384, 14)
    corr_matrix = np.corrcoef(flat.T)  # (14, 14)
    return corr_matrix

corr_matrix = compute_correlation_matrix(eda_files)

# ─── HEATMAP ───────────────────────────────────────────────────────────────
short_names = ['S2-B2\nAzul','S2-B3\nVerde','S2-B4\nRojo','S2-B8\nNIR',
               'S2-B8A\nNIR-A','S2-B11\nSWIR1','S2-B12\nSWIR2',
               'S1-VV\nSAR','S1-VH\nSAR','ALOS\nDEM','DEM\nSlope',
               'S2-B5\nRE1','S2-B6\nRE2','S2-B7\nRE3']

fig, ax = plt.subplots(figsize=(12, 10))
mask_upper = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            xticklabels=short_names, yticklabels=short_names,
            annot_kws={'size': 7.5}, cbar_kws={'label': 'Correlación de Pearson (r)'})

ax.set_title('Matriz de Correlación entre los 14 Canales\nLandslide4Sense', 
             fontsize=13, fontweight='bold', pad=15)
plt.xticks(fontsize=8); plt.yticks(fontsize=8, rotation=0)
plt.tight_layout()
plt.savefig('fig_correlacion.png', dpi=120, bbox_inches='tight')
plt.show()

# ─── TOP CORRELACIONES ─────────────────────────────────────────────────────
pairs = [(corr_matrix[i,j], i, j) 
         for i in range(14) for j in range(i+1, 14)]
pairs.sort(key=lambda x: abs(x[0]), reverse=True)

print("\n🔗 Top 10 pares de mayor correlación absoluta:")
print(f"{'Par':<40} {'r':>8} {'|r|':>8}")
print("-"*58)
for r, i, j in pairs[:10]:
    print(f"  Ch{i:02d} {CHANNEL_NAMES[i]:16s} ↔ Ch{j:02d} {CHANNEL_NAMES[j]:16s}  {r:+.4f}  {abs(r):.4f}")

print("\n⚠️ Pares con r > 0.95 son redundantes — considerar reducción dimensional (PCA)")
high_corr = [(r, i, j) for r, i, j in pairs if abs(r) > 0.95]
print(f"   Canales con correlación |r| > 0.95: {len(high_corr)} pares")

## 6. EDA — Verificación de Leakage y Calidad <a id='leak'></a>

In [ ]:
# ─── CHECK 1: VALORES AUSENTES / NaN ──────────────────────────────────────
print("Verificando valores ausentes en muestra de imágenes...")
nan_counts = {c: 0 for c in range(14)}
zero_ratio  = []

for fp in tqdm(eda_files[:50], desc='Check NaN', leave=False):
    with h5py.File(fp, 'r') as hf:
        arr = hf['img'][:].astype(np.float32)
    for c in range(14):
        nan_counts[c] += np.isnan(arr[:,:,c]).sum()
    zero_ratio.append((arr == 0).mean())

total_nans = sum(nan_counts.values())
print(f"  ✅ Valores NaN: {total_nans} {'(dataset limpio)' if total_nans == 0 else '⚠️ REVISAR'}")
print(f"  ℹ️  Ratio promedio de ceros por parche: {np.mean(zero_ratio)*100:.1f}%")
print(f"  ℹ️  Parches con >30% ceros (posibles bordes de imagen): "
      f"{sum(1 for r in zero_ratio if r > 0.3)}")

In [ ]:
# ─── CHECK 2: DUPLICADOS ──────────────────────────────────────────────────
print("Verificando duplicados (similitud coseno entre parches)...")

n_check = min(100, len(eda_files))
check_files = eda_files[:n_check]
means_per_img = []

for fp in tqdm(check_files, desc='Hash de parches', leave=False):
    with h5py.File(fp, 'r') as hf:
        arr = hf['img'][:].astype(np.float32)
    means_per_img.append(arr.mean(axis=(0, 1)))   # (14,) — fingerprint

M = np.array(means_per_img)   # (n_check, 14)
norms = np.linalg.norm(M, axis=1, keepdims=True)
M_norm = M / (norms + 1e-8)
sim_matrix = M_norm @ M_norm.T   # (n_check, n_check)
np.fill_diagonal(sim_matrix, 0)
max_sim = sim_matrix.max(axis=1)

n_duplicates = (max_sim > 0.995).sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

im = ax1.imshow(sim_matrix, cmap='hot_r', vmin=0, vmax=1)
ax1.set_title(f'Mapa de Similitud Coseno\n({n_check} parches de TestData)', fontsize=11, fontweight='bold')
ax1.set_xlabel('Índice de parche'); ax1.set_ylabel('Índice de parche')
plt.colorbar(im, ax=ax1, label='Similitud coseno')

ax2.hist(max_sim, bins=25, color='steelblue', edgecolor='white', alpha=0.85)
ax2.axvline(0.995, color='red', ls='--', lw=2, label='Umbral duplicados (0.995)')
ax2.set_xlabel('Similitud máxima con otro parche', fontsize=10)
ax2.set_ylabel('Frecuencia', fontsize=10)
ax2.set_title('Distribución de Similitud Máxima\n(control de calidad)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
status = 'green' if n_duplicates == 0 else 'red'
icon = '✅' if n_duplicates == 0 else '⚠️'
ax2.text(0.05, 0.93, f'{icon} Duplicados detectados: {n_duplicates}', 
         transform=ax2.transAxes, fontsize=10, va='top',
         color=status, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightgreen' if n_duplicates == 0 else 'lightyellow'))
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig_leakage_check.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"\n✅ Verificación completada: {n_duplicates} duplicados en {n_check} parches revisados")

In [ ]:
# ─── CHECK 3: BALANCE DE CLASES (solo si TrainData + masks disponibles) ───
if available.get('train') and os.path.exists(TRAIN_MASK_DIR):
    mask_files = sorted(glob.glob(os.path.join(TRAIN_MASK_DIR, '*.h5')))
    labels = []
    
    for mf in tqdm(mask_files, desc='Cargando etiquetas', leave=False):
        with h5py.File(mf, 'r') as hf:
            mask_key = list(hf.keys())[0]
            mask = hf[mask_key][:]
        # Parche positivo si >0% del área es deslizamiento
        labels.append(1 if mask.sum() > 0 else 0)
    
    labels = np.array(labels)
    n_pos = labels.sum()
    n_neg = len(labels) - n_pos
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
    
    # Pie chart
    ax1.pie([n_pos, n_neg], labels=[f'Deslizamiento\n({n_pos} parches)',
                                     f'No-deslizamiento\n({n_neg} parches)'],
            colors=['#e74c3c','#3498db'], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 10})
    ax1.set_title('Balance de Clases — TrainData', fontsize=11, fontweight='bold')
    
    # Area percentage distribution
    areas = []
    for mf in tqdm(mask_files[:200], desc='Área deslizamientos', leave=False):
        with h5py.File(mf, 'r') as hf:
            mask_key = list(hf.keys())[0]
            mask = hf[mask_key][:]
        area_pct = mask.sum() / (128*128) * 100
        if area_pct > 0:
            areas.append(area_pct)
    
    ax2.hist(areas, bins=30, color='#e74c3c', alpha=0.8, edgecolor='white')
    ax2.set_xlabel('% del parche cubierto por deslizamiento', fontsize=10)
    ax2.set_ylabel('Frecuencia', fontsize=10)
    ax2.set_title('Distribución de Área de Deslizamiento\n(parches positivos)', fontsize=11, fontweight='bold')
    ax2.axvline(np.mean(areas), color='navy', ls='--', lw=2, label=f'Media: {np.mean(areas):.1f}%')
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('fig_class_balance.png', dpi=120, bbox_inches='tight')
    plt.show()
    
    print(f"\n📊 Balance de clases:")
    print(f"   Positivos (deslizamiento):    {n_pos:4d} ({100*n_pos/len(labels):.1f}%)")
    print(f"   Negativos (no-deslizamiento): {n_neg:4d} ({100*n_neg/len(labels):.1f}%)")
    print(f"   Ratio de desbalance: 1 : {n_neg/n_pos:.2f}")
    print(f"   ⚠️  El desbalance justifica usar F1-score e IoU como métricas principales (no accuracy)")
else:
    print("ℹ️  TrainData/mask no disponible — análisis de balance requiere máscaras de entrenamiento")
    print("   Descarga el dataset completo de: https://www.kaggle.com/datasets/landslide4sense/competition")

## 7. Preprocesamiento y Augmentation <a id='preprocess'></a>

In [ ]:
# ─── DATASET CLASS ─────────────────────────────────────────────────────────
class Landslide4SenseDataset(Dataset):
    """Dataset PyTorch para Landslide4Sense.
    
    Args:
        img_dir: directorio con archivos .h5 de imágenes
        mask_dir: directorio con archivos .h5 de máscaras (None para test)
        transform: transformaciones Albumentations
        channel_mean: media por canal para normalización (None = calcular)
        channel_std: std por canal para normalización (None = calcular)
        task: 'classification' (etiqueta parche) o 'segmentation' (máscara pixel)
    """
    
    def __init__(self, img_dir, mask_dir=None, transform=None,
                 channel_mean=None, channel_std=None, task='classification'):
        self.img_files  = sorted(glob.glob(os.path.join(img_dir, '*.h5')))
        self.mask_files = sorted(glob.glob(os.path.join(mask_dir, '*.h5'))) if mask_dir else None
        self.transform  = transform
        self.task       = task
        
        # Normalización: calcular sobre muestra si no se proporciona
        if channel_mean is None or channel_std is None:
            self.channel_mean, self.channel_std = self._compute_normalization(n_sample=50)
        else:
            self.channel_mean = np.array(channel_mean)
            self.channel_std  = np.array(channel_std)
    
    def _compute_normalization(self, n_sample=50):
        step = max(1, len(self.img_files) // n_sample)
        sample = self.img_files[::step][:n_sample]
        sum1, sum2, n = np.zeros(14), np.zeros(14), 0
        for fp in sample:
            with h5py.File(fp, 'r') as hf:
                arr = hf['img'][:].astype(np.float32)
            flat = arr.reshape(-1, 14)
            sum1 += flat.sum(0); sum2 += (flat**2).sum(0); n += flat.shape[0]
        mean = sum1 / n
        std  = np.sqrt(np.maximum(sum2/n - mean**2, 1e-8))
        return mean.astype(np.float32), std.astype(np.float32)
    
    def __len__(self):
        return len(self.img_files)
    
    def __getitem__(self, idx):
        with h5py.File(self.img_files[idx], 'r') as hf:
            img = hf['img'][:].astype(np.float32)   # (H, W, 14)
        
        # Normalización z-score por canal
        img = (img - self.channel_mean) / (self.channel_std + 1e-8)  # (H, W, 14)
        
        if self.mask_files:
            with h5py.File(self.mask_files[idx], 'r') as hf:
                mask_key = list(hf.keys())[0]
                mask = hf[mask_key][:].astype(np.float32)   # (H, W)
            
            if self.transform:
                aug = self.transform(image=img, mask=mask)
                img, mask = aug['image'], aug['mask']
            
            if self.task == 'classification':
                label = float(mask.sum() > 0)
                return torch.tensor(img).permute(2,0,1), torch.tensor(label)
            else:
                return torch.tensor(img).permute(2,0,1), torch.tensor(mask).unsqueeze(0)
        else:
            if self.transform:
                aug = self.transform(image=img)
                img = aug['image']
            return torch.tensor(img).permute(2,0,1)

# ─── AUGMENTATION ──────────────────────────────────────────────────────────
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=15, p=0.4, border_mode=0),
    A.GridDistortion(num_steps=5, distort_limit=0.15, p=0.2),
    A.CoarseDropout(max_holes=6, max_height=8, max_width=8,
                    min_holes=2, fill_value=0, p=0.15),
])

val_transform = None   # Sin augmentation en validación/test

print("✅ Clases de Dataset y transforms definidas")
print(f"   train_transform: {len(train_transform.transforms)} augmentaciones")

# ─── PRUEBA RÁPIDA ──────────────────────────────────────────────────────────
if available['test']:
    test_ds = Landslide4SenseDataset(TEST_IMG_DIR, transform=val_transform)
    x_test = test_ds[0]
    print(f"\n✅ Dataset de test cargado: {len(test_ds)} parches")
    print(f"   Shape del tensor: {x_test.shape}")   # (14, 128, 128)
    print(f"   Dtype: {x_test.dtype}")
    print(f"   Media (canal 0, post-norm): {x_test[0].mean():.4f}  (≈ 0 si normalización correcta)")

## 8. Definición de Modelos <a id='models'></a>

In [ ]:
# ─── ADAPTADOR DE ENTRADA PARA 14 CANALES ─────────────────────────────────
def adapt_first_conv(model, n_channels=14, strategy='mean'):
    """Adapta la primera capa conv de cualquier modelo ImageNet a n_channels.
    
    Estrategia 'mean': replica los pesos RGB y promedia para canales adicionales.
    Esta estrategia preserva el conocimiento preentrenado mejor que inicialización aleatoria.
    """
    # Encontrar primera capa conv
    first_conv = None
    first_conv_name = None
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d) and module.in_channels == 3:
            first_conv = module
            first_conv_name = name
            break
    
    if first_conv is None:
        raise ValueError("No se encontró primera capa Conv2d con 3 canales")
    
    # Crear nueva conv con n_channels
    new_conv = nn.Conv2d(
        n_channels, first_conv.out_channels,
        kernel_size=first_conv.kernel_size, stride=first_conv.stride,
        padding=first_conv.padding, bias=first_conv.bias is not None
    )
    
    # Inicializar: replicar pesos originales para los 3 primeros canales
    with torch.no_grad():
        if strategy == 'mean':
            w = first_conv.weight.data   # (out_ch, 3, kH, kW)
            mean_w = w.mean(dim=1, keepdim=True).repeat(1, n_channels, 1, 1) / n_channels
            new_conv.weight.data = mean_w
            # Sobreescribir los primeros 3 canales con pesos originales
            new_conv.weight.data[:, :3, :, :] = w
    
    # Reemplazar en el modelo
    parts = first_conv_name.split('.')
    parent = model
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new_conv)
    
    return model

# ─── MODELO 1: ResNet-50 Fine-Tuned ────────────────────────────────────────
def build_resnet50(n_channels=14, pretrained=True, freeze_backbone_epochs=10):
    weights = tv_models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
    model = tv_models.resnet50(weights=weights)
    model = adapt_first_conv(model, n_channels)
    
    # Reemplazar cabeza de clasificación
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 1)
    )
    return model

# ─── MODELO 2: EfficientNet-B4 Fine-Tuned ─────────────────────────────────
def build_efficientnet_b4(n_channels=14, pretrained=True):
    weights = tv_models.EfficientNet_B4_Weights.IMAGENET1K_V1 if pretrained else None
    model = tv_models.efficientnet_b4(weights=weights)
    model = adapt_first_conv(model, n_channels)
    
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 1)
    )
    return model

# ─── MODELO 3: U-Net + ResNet-34 (segmentación) ────────────────────────────
try:
    import segmentation_models_pytorch as smp
    
    def build_unet_resnet34(n_channels=14, pretrained=True):
        encoder_weights = 'imagenet' if pretrained else None
        model = smp.Unet(
            encoder_name='resnet34',
            encoder_weights=encoder_weights,
            in_channels=n_channels,
            classes=1,
            activation=None
        )
        return model
    HAS_SMP = True
    print("✅ segmentation-models-pytorch disponible")
except ImportError:
    HAS_SMP = False
    print("⚠️  segmentation-models-pytorch no disponible (pip install segmentation-models-pytorch)")

# Verificar parámetros
r50  = build_resnet50(pretrained=False)
eff4 = build_efficientnet_b4(pretrained=False)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Parámetros entrenables:")
print(f"   ResNet-50:         {count_params(r50):>12,}")
print(f"   EfficientNet-B4:   {count_params(eff4):>12,}")
if HAS_SMP:
    unet = build_unet_resnet34(pretrained=False)
    print(f"   U-Net+ResNet-34:   {count_params(unet):>12,}")

In [ ]:
# ─── FUNCIÓN DE ENTRENAMIENTO GENÉRICA ─────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    
    for batch in tqdm(loader, desc='  Train', leave=False):
        x, y = batch[0].to(device), batch[1].to(device).float()
        optimizer.zero_grad()
        logits = model(x).squeeze(-1)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(y)
        preds = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(y.cpu().numpy().tolist())
    
    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(all_labels, (np.array(all_preds) > 0.5).astype(int), zero_division=0)
    return avg_loss, f1

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    
    for batch in tqdm(loader, desc='  Eval ', leave=False):
        x, y = batch[0].to(device), batch[1].to(device).float()
        logits = model(x).squeeze(-1)
        loss = criterion(logits, y)
        
        total_loss += loss.item() * len(y)
        preds = torch.sigmoid(logits).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(y.cpu().numpy().tolist())
    
    avg_loss = total_loss / len(loader.dataset)
    probs = np.array(all_preds)
    labels = np.array(all_labels)
    
    # Optimizar umbral sobre conjunto de validación
    best_f1, best_thr = 0, 0.5
    for thr in np.arange(0.3, 0.7, 0.05):
        f1 = f1_score(labels, (probs > thr).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    
    auc = roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.5
    return avg_loss, best_f1, auc, best_thr, probs, labels

print("✅ Funciones de entrenamiento definidas")

## 9. Entrenamiento con 5-Fold Cross-Validation <a id='train'></a>

In [ ]:
# ─── CONFIGURACIÓN DEL EXPERIMENTO ────────────────────────────────────────
# ⚠️  REQUIERE TrainData completo con máscaras para ejecutar el entrenamiento
# Si solo tienes TestData, este bloque mostrará las instrucciones de descarga

TRAIN_AVAILABLE = available.get('train') and os.path.exists(TRAIN_MASK_DIR)

if not TRAIN_AVAILABLE:
    print("⚠️  TrainData no disponible. Para ejecutar el entrenamiento:")
    print("   1. Descarga el dataset completo: https://www.kaggle.com/datasets/landslide4sense/competition")
    print("   2. Extrae en DATA_ROOT con la estructura TrainData/img/ + TrainData/mask/")
    print("   3. Ejecuta nuevamente este bloque")
    print()
    print("   Mientras tanto, el código completo de entrenamiento está disponible abajo.")
else:
    print(f"✅ TrainData disponible: {len(available['train'])} imágenes + máscaras")
    print("   Ejecutar las celdas siguientes para iniciar el entrenamiento")

In [ ]:
# ─── 5-FOLD CROSS-VALIDATION ────────────────────────────────────────────────
# (ejecutar solo si TRAIN_AVAILABLE = True)

EXPERIMENT_CONFIG = {
    'n_folds': 5,
    'batch_size': 16,
    'n_epochs': 50,        # Reducir para pruebas; 80-100 para producción
    'patience': 10,
    'lr_head': 1e-4,
    'lr_backbone': 1e-5,
    'weight_decay': 1e-4,
    'n_workers': 2,
    'save_dir': 'checkpoints',
}
os.makedirs(EXPERIMENT_CONFIG['save_dir'], exist_ok=True)

def run_kfold_experiment(model_name, model_fn, config=EXPERIMENT_CONFIG):
    """Ejecuta 5-fold CV y retorna métricas por fold.""""
    if not TRAIN_AVAILABLE:
        print(f"⚠️  Entrenamiento de {model_name} omitido — TrainData no disponible")
        return None
    
    # Recolectar labels para estratificación
    print(f"\n{'='*60}")
    print(f"🚀 Iniciando {config['n_folds']}-Fold CV: {model_name}")
    print(f"{'='*60}")
    
    mask_files = sorted(glob.glob(os.path.join(TRAIN_MASK_DIR, '*.h5')))
    labels = []
    for mf in tqdm(mask_files, desc='Cargando labels', leave=False):
        with h5py.File(mf, 'r') as hf:
            mk = list(hf.keys())[0]
            labels.append(1 if hf[mk][:].sum() > 0 else 0)
    labels = np.array(labels)
    
    skf = StratifiedKFold(n_splits=config['n_folds'], shuffle=True, random_state=SEED)
    fold_metrics = []
    
    # Calcular normalización sobre todo el training set
    full_train_ds = Landslide4SenseDataset(TRAIN_IMG_DIR, transform=None)
    ch_mean, ch_std = full_train_ds.channel_mean, full_train_ds.channel_std
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(available['train'], labels)):
        print(f"\n--- Fold {fold+1}/{config['n_folds']} ---")
        
        # Crear datasets del fold
        train_files = [available['train'][i] for i in train_idx]
        val_files   = [available['train'][i] for i in val_idx]
        train_masks = [mask_files[i] for i in train_idx]
        val_masks   = [mask_files[i] for i in val_idx]
        
        # Crear datasets temporales con rutas del fold
        # (simplificado: crear clase auxiliar)
        class FoldDataset(Dataset):
            def __init__(self, img_files, mask_files, transform, mean, std):
                self.imgs  = img_files
                self.masks = mask_files
                self.tf    = transform
                self.mean  = mean
                self.std   = std
            
            def __len__(self): return len(self.imgs)
            
            def __getitem__(self, idx):
                with h5py.File(self.imgs[idx],'r') as hf:
                    img = hf['img'][:].astype(np.float32)
                with h5py.File(self.masks[idx],'r') as hf:
                    mk = list(hf.keys())[0]
                    mask = hf[mk][:].astype(np.float32)
                img = (img - self.mean) / (self.std + 1e-8)
                if self.tf:
                    aug = self.tf(image=img, mask=mask)
                    img, mask = aug['image'], aug['mask']
                label = float(mask.sum() > 0)
                return torch.tensor(img).permute(2,0,1).float(), torch.tensor(label).float()
        
        train_ds = FoldDataset(train_files, train_masks, train_transform, ch_mean, ch_std)
        val_ds   = FoldDataset(val_files,   val_masks,   val_transform,   ch_mean, ch_std)
        
        train_loader = DataLoader(train_ds, batch_size=config['batch_size'],
                                  shuffle=True,  num_workers=config['n_workers'],
                                  pin_memory=True, drop_last=True)
        val_loader   = DataLoader(val_ds,   batch_size=config['batch_size'],
                                  shuffle=False, num_workers=config['n_workers'],
                                  pin_memory=True)
        
        # Modelo
        model = model_fn().to(DEVICE)
        
        # Optimizer con learning rates diferenciados
        backbone_params = [p for n,p in model.named_parameters()
                           if not any(x in n for x in ['fc','classifier','head'])]
        head_params     = [p for n,p in model.named_parameters()
                           if any(x in n for x in ['fc','classifier','head'])]
        
        optimizer = optim.AdamW([
            {'params': backbone_params, 'lr': config['lr_backbone']},
            {'params': head_params,     'lr': config['lr_head']},
        ], weight_decay=config['weight_decay'])
        
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=config['n_epochs'])
        
        # Pérdida ponderada (peso para clase positiva)
        n_neg = (labels[train_idx] == 0).sum()
        n_pos = (labels[train_idx] == 1).sum()
        pos_weight = torch.tensor([n_neg / max(n_pos, 1)]).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        
        # ─── Loop de entrenamiento ─────────────────────────────────────────
        best_f1, best_auc, patience_count = 0, 0, 0
        history = {'train_loss':[], 'val_loss':[], 'train_f1':[], 'val_f1':[], 'val_auc':[]}
        
        # Fase 1: congelar backbone (solo cabeza) — primeras 5 épocas
        for param in backbone_params:
            param.requires_grad = False
        
        for epoch in range(config['n_epochs']):
            # Descongelar backbone después de 5 épocas
            if epoch == 5:
                for param in backbone_params:
                    param.requires_grad = True
                print(f"   Época {epoch}: backbone descongelado ✓")
            
            train_loss, train_f1 = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
            val_loss, val_f1, val_auc, best_thr, _, _ = eval_epoch(model, val_loader, criterion, DEVICE)
            scheduler.step()
            
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['train_f1'].append(train_f1)
            history['val_f1'].append(val_f1)
            history['val_auc'].append(val_auc)
            
            if (epoch + 1) % 5 == 0:
                print(f"   Ep {epoch+1:3d}: train_loss={train_loss:.4f} "
                      f"val_F1={val_f1:.4f} val_AUC={val_auc:.4f} "
                      f"[thr={best_thr:.2f}]")
            
            # Early stopping
            if val_f1 > best_f1:
                best_f1, best_auc = val_f1, val_auc
                patience_count = 0
                torch.save(model.state_dict(),
                           os.path.join(config['save_dir'], f'{model_name}_fold{fold+1}_best.pt'))
            else:
                patience_count += 1
                if patience_count >= config['patience']:
                    print(f"   Early stopping en época {epoch+1}")
                    break
        
        # Evaluar con el mejor modelo guardado
        model.load_state_dict(torch.load(
            os.path.join(config['save_dir'], f'{model_name}_fold{fold+1}_best.pt'),
            map_location=DEVICE))
        _, val_f1_final, val_auc_final, best_thr, probs, true_labels = eval_epoch(
            model, val_loader, criterion, DEVICE)
        
        preds_final = (probs > best_thr).astype(int)
        prec  = precision_score(true_labels, preds_final, zero_division=0)
        rec   = recall_score(true_labels, preds_final, zero_division=0)
        
        fold_result = {
            'fold': fold+1, 'f1': val_f1_final, 'auc': val_auc_final,
            'precision': prec, 'recall': rec, 'threshold': best_thr,
            'history': history
        }
        fold_metrics.append(fold_result)
        print(f"   ✅ Fold {fold+1} final: F1={val_f1_final:.4f} | AUC={val_auc_final:.4f} | "
              f"P={prec:.4f} | R={rec:.4f}")
    
    return fold_metrics

# ─── EJECUTAR EXPERIMENTOS ──────────────────────────────────────────────────
all_results = {}

if TRAIN_AVAILABLE:
    print("Iniciando entrenamiento...")
    all_results['ResNet50_FT']  = run_kfold_experiment('ResNet50_FT',  
                                      lambda: build_resnet50(pretrained=True))
    all_results['EffNet_B4_FT'] = run_kfold_experiment('EffNetB4_FT',  
                                      lambda: build_efficientnet_b4(pretrained=True))
    all_results['ResNet50_Scratch'] = run_kfold_experiment('ResNet50_Scratch',
                                      lambda: build_resnet50(pretrained=False))
else:
    print("⚠️  Entrenamiento requiere TrainData completo.")
    print("   El EDA y el código están listos. Descarga el dataset para continuar.")

## 10. Evaluación y Visualización de Resultados <a id='eval'></a>

In [ ]:
# ─── TABLA COMPARATIVA DE RESULTADOS ──────────────────────────────────────
def summarize_results(all_results):
    rows = []
    for model_name, fold_metrics in all_results.items():
        if fold_metrics is None:
            continue
        f1s  = [m['f1']  for m in fold_metrics]
        aucs = [m['auc'] for m in fold_metrics]
        precs = [m['precision'] for m in fold_metrics]
        recs  = [m['recall']    for m in fold_metrics]
        rows.append({
            'Modelo':     model_name,
            'F1 (media)': f'{np.mean(f1s):.4f}',
            'F1 (±std)':  f'{np.std(f1s):.4f}',
            'AUC (media)': f'{np.mean(aucs):.4f}',
            'AUC (±std)':  f'{np.std(aucs):.4f}',
            'Precisión':   f'{np.mean(precs):.4f}',
            'Recall':      f'{np.mean(recs):.4f}',
        })
    return pd.DataFrame(rows) if rows else None

if all_results:
    summary = summarize_results(all_results)
    if summary is not None:
        print("\n📊 Tabla Comparativa Final (5-Fold CV):")
        print(summary.to_string(index=False))
        summary.to_csv('resultados_comparativos.csv', index=False)
        print("\n✅ Tabla guardada en resultados_comparativos.csv")
else:
    print("ℹ️  No hay resultados de entrenamiento disponibles aún.")
    print("   Ejecuta la Sección 9 con TrainData completo para ver la tabla.")

In [ ]:
# ─── CURVAS DE APRENDIZAJE ─────────────────────────────────────────────────
def plot_training_curves(all_results, model_name):
    if model_name not in all_results or all_results[model_name] is None:
        print(f"Sin datos para {model_name}")
        return
    
    fold_metrics = all_results[model_name]
    n_folds = len(fold_metrics)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f'Curvas de Entrenamiento — {model_name}\n(5 Folds)', fontsize=12, fontweight='bold')
    
    colors = plt.cm.Set1(np.linspace(0, 0.8, n_folds))
    
    for fold_idx, m in enumerate(fold_metrics):
        hist = m['history']
        epochs = range(1, len(hist['train_loss'])+1)
        c = colors[fold_idx]
        
        axes[0].plot(epochs, hist['train_loss'], '-', color=c, alpha=0.7, lw=1.5)
        axes[0].plot(epochs, hist['val_loss'], '--', color=c, alpha=0.7, lw=1.5,
                     label=f'F{fold_idx+1} val')
        
        axes[1].plot(epochs, hist['train_f1'], '-', color=c, alpha=0.7, lw=1.5)
        axes[1].plot(epochs, hist['val_f1'], '--', color=c, alpha=0.7, lw=1.5,
                     label=f'F{fold_idx+1}')
        
        axes[2].plot(epochs, hist['val_auc'], '-', color=c, lw=1.5,
                     label=f'F{fold_idx+1}: AUC={m["auc"]:.3f}')
    
    for ax, ylabel, title in zip(axes,
        ['Loss', 'F1-score', 'AUC-ROC'],
        ['Pérdida de Entrenamiento vs. Validación',
         'F1-score por Época',
         'AUC-ROC de Validación']):
        ax.set_xlabel('Época', fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'fig_curvas_{model_name}.png', dpi=120, bbox_inches='tight')
    plt.show()

for model_name in all_results:
    plot_training_curves(all_results, model_name)

# ─── CURVA ROC COMPARATIVA ────────────────────────────────────────────────
print("Curvas ROC serán graficadas con los resultados del conjunto de prueba final.")
print("\n✅ Pipeline completo. Resumen del flujo:")
print("   1. EDA ✓  → estadísticas por canal, histogramas, correlación, leakage")
print("   2. Preprocesamiento ✓  → normalización z-score + augmentation")
print("   3. Modelos ✓  → ResNet-50, EfficientNet-B4, U-Net+ResNet-34")
print("   4. Entrenamiento ✓  → 5-fold CV, early stopping, umbral optimizado")
print("   5. Evaluación ✓  → F1, AUC-ROC, Precisión, Recall")